# Silver Layer: Data Cleaning and Standardization

This notebook transforms the Bronze dataset into a cleaned and analytics ready Silver layer.

Key transformations include:
- data type standardization
- missing value handling
- decoding categorical variables
- creating derived analytical metrics

## Pipeline Architecture Context

This project follows a layered data pipeline design inspired by modern lakehouse architectures used in platforms such as Microsoft Fabric and Databricks.

Data processing is separated into three stages:

**Bronze Layer**
- Raw data ingestion from the source dataset
- Minimal transformation
- Preservation of the original dataset for reproducibility

**Silver Layer**
- Data cleaning and standardization
- Type conversion and missing value handling
- Decoding categorical variables into business-readable fields
- Creation of derived analytical metrics

Separating these layers ensures that transformations are reproducible and that raw data remains unchanged. The Silver layer therefore reloads the Bronze dataset and applies standardized transformations before producing analytics ready outputs.

**Gold Layer**
- Aggregated analytical tables designed for dashboards and reporting.

In [1]:
import pandas as pd

bronze_df = pd.read_csv("../data_raw/college_scorecard.csv", low_memory=False)

print(bronze_df.shape)

bronze_df.head()

(6429, 3306)


,UNITID,OPEID,OPEID6,INSTNM,CITY,STABBR,ZIP,ACCREDAGENCY,INSTURL,NPCURL,...,COUNT_WNE_MALE1_P11,GT_THRESHOLD_P11,MD_EARN_WNE_INC1_P11,MD_EARN_WNE_INC2_P11,MD_EARN_WNE_INC3_P11,MD_EARN_WNE_INDEP0_P11,MD_EARN_WNE_INDEP1_P11,MD_EARN_WNE_MALE0_P11,MD_EARN_WNE_MALE1_P11,SCORECARD_SECTOR
0,100654,100200.0,1002.0,Alabama A & M University,Normal,AL,35762,Southern Association of Colleges and Schools C...,www.aamu.edu/,www.aamu.edu/admissions-aid/tuition-fees/net-p...,...,777.0,0.6250,36650.0,41070.0,47016.0,38892.0,41738.0,38167.0,40250.0,4
1,100663,105200.0,1052.0,University of Alabama at Birmingham,Birmingham,AL,35294-0110,Southern Association of Colleges and Schools C...,https://www.uab.edu/,https://tcc.ruffalonl.com/University of Alabam...,...,1157.0,0.7588,47182.0,51896.0,54368.0,50488.0,51505.0,46559.0,59181.0,4
2,100690,2503400.0,25034.0,Amridge University,Montgomery,AL,36117-3553,Southern Association of Colleges and Schools C...,https://www.amridgeuniversity.edu/,https://www2.amridgeuniversity.edu:9091/,...,67.0,0.5986,35752.0,41007.0,NaN,NaN,38467.0,32654.0,49435.0,5
3,100706,105500.0,1055.0,University of Alabama in Huntsville,Huntsville,AL,35899,Southern Association of Colleges and Schools C...,www.uah.edu/,uah.clearcostcalculator.com/student/default/ne...,...,802.0,0.7810,51208.0,62219.0,62577.0,55920.0,60221.0,47787.0,67454.0,4
4,100724,100500.0,1005.0,Alabama State University,Montgomery,AL,36104-0271,Southern Association of Colleges and Schools C...,www.alasu.edu/,tcc.ruffalonl.com/Alabama State University/Fre...,...,1049.0,0.5378,32844.0,36932.0,37966.0,34294.0,31797.0,32303.0,36964.0,4


In [2]:
selected_columns = [
    "UNITID",
    "INSTNM",
    "CITY",
    "STABBR",
    "CONTROL",
    "LOCALE",
    "UGDS",
    "TUITIONFEE_IN",
    "TUITIONFEE_OUT",
    "C150_4",
    "MD_EARN_WNE_P10"
]

silver_df = bronze_df[selected_columns].copy()

print("Silver dataset shape:", silver_df.shape)

silver_df.head()

Silver dataset shape: (6429, 11)


,UNITID,INSTNM,CITY,STABBR,CONTROL,LOCALE,UGDS,TUITIONFEE_IN,TUITIONFEE_OUT,C150_4,MD_EARN_WNE_P10
0,100654,Alabama A & M University,Normal,AL,1,12.0,5726.0,10024.0,18634.0,0.2874,40628.0
1,100663,University of Alabama at Birmingham,Birmingham,AL,1,12.0,12118.0,8832.0,21864.0,0.6260,54501.0
2,100690,Amridge University,Montgomery,AL,2,12.0,226.0,NaN,NaN,0.4000,37621.0
3,100706,University of Alabama in Huntsville,Huntsville,AL,1,12.0,6650.0,11770.0,24662.0,0.6191,61767.0
4,100724,Alabama State University,Montgomery,AL,1,12.0,3322.0,11248.0,19576.0,0.3018,34502.0


### Convert Numeric Columns:

In [3]:
numeric_columns = [
    "UGDS",
    "TUITIONFEE_IN",
    "TUITIONFEE_OUT",
    "C150_4",
    "MD_EARN_WNE_P10"
]

for col in numeric_columns:
    silver_df[col] = pd.to_numeric(silver_df[col], errors="coerce")

silver_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6429 entries, 0 to 6428
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   UNITID           6429 non-null   int64  
 1   INSTNM           6429 non-null   object 
 2   CITY             6429 non-null   object 
 3   STABBR           6429 non-null   object 
 4   CONTROL          6429 non-null   int64  
 5   LOCALE           5924 non-null   float64
 6   UGDS             5656 non-null   float64
 7   TUITIONFEE_IN    3729 non-null   float64
 8   TUITIONFEE_OUT   3729 non-null   float64
 9   C150_4           2272 non-null   float64
 10  MD_EARN_WNE_P10  5280 non-null   float64
dtypes: float64(6), int64(2), object(3)
memory usage: 552.6+ KB


### Decode Institution Type:

In [4]:
control_map = {
    1: "Public",
    2: "Private nonprofit",
    3: "Private for-profit"
}

silver_df["CONTROL_DESC"] = silver_df["CONTROL"].map(control_map)

silver_df[["CONTROL", "CONTROL_DESC"]].head()

,CONTROL,CONTROL_DESC
0,1,Public
1,1,Public
2,2,Private nonprofit
3,1,Public
4,1,Public


### Decode Location Type:

In [5]:
silver_df["LOCALE"] = silver_df["LOCALE"].astype(str)

silver_df["LOCALE_GROUP"] = silver_df["LOCALE"].str[0].map({
    "1": "City",
    "2": "Suburb",
    "3": "Town",
    "4": "Rural"
})

silver_df[["LOCALE", "LOCALE_GROUP"]].head()

,LOCALE,LOCALE_GROUP
0,12.0,City
1,12.0,City
2,12.0,City
3,12.0,City
4,12.0,City


### Create Business Metrics: 

This metric shows return on education investment, which will be extremely useful for analytics.

In [6]:
silver_df["AVG_TUITION"] = silver_df[["TUITIONFEE_IN", "TUITIONFEE_OUT"]].mean(axis=1)

silver_df["EARNINGS_TO_TUITION_RATIO"] = (
    silver_df["MD_EARN_WNE_P10"] / silver_df["AVG_TUITION"]
)

silver_df[[
    "INSTNM",
    "AVG_TUITION",
    "MD_EARN_WNE_P10",
    "EARNINGS_TO_TUITION_RATIO"
]].head()

,INSTNM,AVG_TUITION,MD_EARN_WNE_P10,EARNINGS_TO_TUITION_RATIO
0,Alabama A & M University,14329.0,40628.0,2.835369
1,University of Alabama at Birmingham,15348.0,54501.0,3.551016
2,Amridge University,NaN,37621.0,NaN
3,University of Alabama in Huntsville,18216.0,61767.0,3.390810
4,Alabama State University,15412.0,34502.0,2.238645


## Pipeline Architecture Context

This project follows a layered data pipeline design inspired by modern lakehouse architectures used in platforms such as Microsoft Fabric and Databricks.

Data processing is separated into three stages:

**Bronze Layer**
- Raw data ingestion from the source dataset
- Minimal transformation
- Preservation of the original dataset for reproducibility

**Silver Layer**
- Data cleaning and standardization
- Type conversion and missing value handling
- Decoding categorical variables into businessr eadable fields
- Creation of derived analytical metrics

Separating these layers ensures that transformations are reproducible and that raw data remains unchanged. The Silver layer therefore reloads the Bronze dataset and applies standardized transformations before producing analytics ready outputs.

**Gold Layer**
- Aggregated analytical tables designed for dashboards and reporting.

### Shape and Columns Check

In [7]:
print("Silver shape:", silver_df.shape)
print("\nColumns:")
print(silver_df.columns.tolist())

Silver shape: (6429, 15)

Columns:
['UNITID', 'INSTNM', 'CITY', 'STABBR', 'CONTROL', 'LOCALE', 'UGDS', 'TUITIONFEE_IN', 'TUITIONFEE_OUT', 'C150_4', 'MD_EARN_WNE_P10', 'CONTROL_DESC', 'LOCALE_GROUP', 'AVG_TUITION', 'EARNINGS_TO_TUITION_RATIO']


### Data Types Check:

In [8]:
silver_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6429 entries, 0 to 6428
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   UNITID                     6429 non-null   int64  
 1   INSTNM                     6429 non-null   object 
 2   CITY                       6429 non-null   object 
 3   STABBR                     6429 non-null   object 
 4   CONTROL                    6429 non-null   int64  
 5   LOCALE                     6429 non-null   object 
 6   UGDS                       5656 non-null   float64
 7   TUITIONFEE_IN              3729 non-null   float64
 8   TUITIONFEE_OUT             3729 non-null   float64
 9   C150_4                     2272 non-null   float64
 10  MD_EARN_WNE_P10            5280 non-null   float64
 11  CONTROL_DESC               6429 non-null   object 
 12  LOCALE_GROUP               5921 non-null   object 
 13  AVG_TUITION                3729 non-null   float

### Null Summary Check:

In [9]:
silver_df.isnull().sum().sort_values(ascending=False)

C150_4                       4157
EARNINGS_TO_TUITION_RATIO    2957
TUITIONFEE_IN                2700
TUITIONFEE_OUT               2700
AVG_TUITION                  2700
MD_EARN_WNE_P10              1149
UGDS                          773
LOCALE_GROUP                  508
UNITID                          0
INSTNM                          0
CITY                            0
STABBR                          0
CONTROL                         0
LOCALE                          0
CONTROL_DESC                    0
dtype: int64

### CONTROL decoding check:

In [10]:
silver_df[["CONTROL", "CONTROL_DESC"]].drop_duplicates().sort_values("CONTROL")

,CONTROL,CONTROL_DESC
0,1,Public
2,2,Private nonprofit
12,3,Private for-profit


### LOCALE grouping check:

In [11]:
silver_df[["LOCALE", "LOCALE_GROUP"]].drop_duplicates().sort_values("LOCALE").head(15)

,LOCALE,LOCALE_GROUP
3576,-3.0,NaN
55,11.0,City
0,12.0,City
9,13.0,City
11,21.0,Suburb
185,22.0,Suburb
26,23.0,Suburb
7,31.0,Town
6,32.0,Town
51,33.0,Town


### Derived Metric check:

In [12]:
silver_df[[
    "INSTNM",
    "AVG_TUITION",
    "MD_EARN_WNE_P10",
    "EARNINGS_TO_TUITION_RATIO"
]].describe()

,AVG_TUITION,MD_EARN_WNE_P10,EARNINGS_TO_TUITION_RATIO
count,3729.000000,5280.000000,3472.000000
mean,18893.052025,43508.301136,3.920179
std,14906.960404,17033.197929,3.230639
min,600.000000,8579.000000,0.463727
25%,7864.500000,31830.000000,1.839338
50%,13920.000000,40567.500000,3.128484
75%,24493.000000,51994.000000,5.317454
max,69330.000000,143372.000000,95.883598


### Infinite Values Check:

In [13]:
import numpy as np

np.isinf(silver_df["EARNINGS_TO_TUITION_RATIO"]).sum()

0

### Duplicate Instituition Check:

In [14]:
silver_df["UNITID"].duplicated().sum()

0

### Business Validation Sample:

In [15]:
silver_df[[
    "INSTNM",
    "CONTROL_DESC",
    "LOCALE_GROUP",
    "UGDS",
    "AVG_TUITION",
    "C150_4",
    "MD_EARN_WNE_P10",
    "EARNINGS_TO_TUITION_RATIO"
]].head(10)

,INSTNM,CONTROL_DESC,LOCALE_GROUP,UGDS,AVG_TUITION,C150_4,MD_EARN_WNE_P10,EARNINGS_TO_TUITION_RATIO
0,Alabama A & M University,Public,City,5726.0,14329.0,0.2874,40628.0,2.835369
1,University of Alabama at Birmingham,Public,City,12118.0,15348.0,0.6260,54501.0,3.551016
2,Amridge University,Private nonprofit,City,226.0,NaN,0.4000,37621.0,NaN
3,University of Alabama in Huntsville,Public,City,6650.0,18216.0,0.6191,61767.0,3.390810
4,Alabama State University,Public,City,3322.0,15412.0,0.3018,34502.0,2.238645
5,The University of Alabama,Public,City,32323.0,22550.0,0.7369,59221.0,2.626208
6,Central Alabama Community College,Public,Town,1134.0,6945.0,NaN,33506.0,4.824478
7,Athens State University,Public,Town,2407.0,NaN,NaN,50273.0,NaN
8,Auburn University at Montgomery,Public,City,2693.0,14764.0,0.3568,44391.0,3.006705
9,Auburn University,Public,City,25732.0,23240.0,0.7921,65337.0,2.811403


### Analytics Readiness Flag

In the Silver layer we introduce a data quality indicator that identifies whether an institution record contains sufficient information for downstream analytical use.

Many institutions in the College Scorecard dataset have missing values for key metrics such as tuition, graduation rate, or post-graduation earnings. Rather than removing these rows during cleaning, we preserve them but flag whether they are suitable for analysis.

A new column `ANALYTICS_READY` is created using the following business rule:

A record is considered analytics-ready if the following fields are present:
- `AVG_TUITION`
- `MD_EARN_WNE_P10`
- `C150_4`

If any of these fields are missing, the record is marked as not ready for analysis.

This approach ensures that:
- No raw data is lost during transformation
- Downstream analytical workflows can easily filter valid records
- Data quality is explicitly documented within the pipeline

In [16]:
# Define fields required for analytical use
required_columns = [
    "AVG_TUITION",
    "MD_EARN_WNE_P10",
    "C150_4"
]

# Create analytics readiness flag
silver_df["ANALYTICS_READY"] = silver_df[required_columns].notnull().all(axis=1)

# Check distribution
silver_df["ANALYTICS_READY"].value_counts()

ANALYTICS_READY
False    4372
True     2057
Name: count, dtype: int64

In [17]:
# Inspect records with the readiness flag

silver_df[
    [
        "INSTNM",
        "AVG_TUITION",
        "MD_EARN_WNE_P10",
        "C150_4",
        "ANALYTICS_READY"
    ]
].head(10)

,INSTNM,AVG_TUITION,MD_EARN_WNE_P10,C150_4,ANALYTICS_READY
0,Alabama A & M University,14329.0,40628.0,0.2874,True
1,University of Alabama at Birmingham,15348.0,54501.0,0.6260,True
2,Amridge University,NaN,37621.0,0.4000,False
3,University of Alabama in Huntsville,18216.0,61767.0,0.6191,True
4,Alabama State University,15412.0,34502.0,0.3018,True
5,The University of Alabama,22550.0,59221.0,0.7369,True
6,Central Alabama Community College,6945.0,33506.0,NaN,False
7,Athens State University,NaN,50273.0,NaN,False
8,Auburn University at Montgomery,14764.0,44391.0,0.3568,True
9,Auburn University,23240.0,65337.0,0.7921,True


In [18]:
# Confirm dataset shape after adding the new column

print("Final Silver shape:", silver_df.shape)
print("\nColumns:")
print(silver_df.columns.tolist())

Final Silver shape: (6429, 16)

Columns:
['UNITID', 'INSTNM', 'CITY', 'STABBR', 'CONTROL', 'LOCALE', 'UGDS', 'TUITIONFEE_IN', 'TUITIONFEE_OUT', 'C150_4', 'MD_EARN_WNE_P10', 'CONTROL_DESC', 'LOCALE_GROUP', 'AVG_TUITION', 'EARNINGS_TO_TUITION_RATIO', 'ANALYTICS_READY']


### Final Silver Dataset Output

At this stage the Silver dataset has completed all cleaning and transformation steps.  
The dataset now contains standardized schema, decoded categorical variables, and derived analytical metrics.

Key transformations applied in the Silver layer include:

- Selection of core analytical columns from the raw College Scorecard dataset
- Conversion of numeric fields to appropriate data types
- Decoding of categorical variables (`CONTROL` → `CONTROL_DESC`)
- Grouping of locale classifications (`LOCALE` → `LOCALE_GROUP`)
- Creation of derived metrics:
  - `AVG_TUITION`
  - `EARNINGS_TO_TUITION_RATIO`
- Introduction of a data quality indicator (`ANALYTICS_READY`) to identify records with sufficient information for analysis

The resulting dataset contains 6429 institutions and 16 curated columns and is now ready for downstream analytical modeling in the Gold layer.

The cleaned dataset is saved to the `data_processed` directory for use in subsequent notebooks.

In [19]:
silver_df.to_csv("../data_processed/silver_college_scorecard.csv", index=False)
print("Updated silver dataset saved.")

Updated silver dataset saved.
